# NTSC v3 Feature Extraction — CIFAR-10

Load the saved NTSC v3 CNN and extract aligned train, validation, and official test embeddings. This notebook does not fit models or score test accuracy.


## 1. Import dan Reproducibility

`SEED = 30092026` harus sama dengan RGB dan NTSC Training supaya sample alignment tetap konsisten.

In [1]:
import os
import random
import numpy as np

SEED = 30092026

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf

# Lebih aman untuk GPU dengan VRAM terbatas.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

tf.random.set_seed(SEED)

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model, load_model
from sklearn.model_selection import train_test_split

print("SEED:", SEED)
print("GPU :", gpus)

I0000 00:00:1791417939.163816    5182 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791417939.596549    5182 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791417941.572532    5182 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


SEED: 30092026
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Konfigurasi File

In [2]:
CNN_MODEL_PATH = "models/cifar10_ntsc_cnn_v3.keras"
FEATURE_OUTPUT_PATH = "features/ntsc_features_v3.npz"
FEATURE_LAYER = "svm_features"

if not os.path.exists(CNN_MODEL_PATH):
    raise FileNotFoundError(
        f"{CNN_MODEL_PATH} tidak ditemukan. "
        "Jalankan NTSC Training terlebih dahulu dan letakkan file model "
        "pada folder yang sama dengan notebook ini."
    )

print("CNN model     :", CNN_MODEL_PATH)
print("Feature output:", FEATURE_OUTPUT_PATH)
print("Feature layer :", FEATURE_LAYER)

CNN model     : models/cifar10_ntsc_cnn_v3.keras
Feature output: features/ntsc_features_v3.npz
Feature layer : svm_features


## 3. Load CIFAR-10, Grayscale NTSC, dan Recreate Split

Split dibuat dengan `SEED = 30092026` dan stratification yang sama.

Setelah split ditentukan, RGB images dikonversi menjadi:

```text
NTSC = 0.299R + 0.587G + 0.114B
```

`train_indices` dan `val_indices` disimpan agar alignment RGB, AVG, dan NTSC dapat diverifikasi.

In [3]:
def rgb_to_ntsc_grayscale(images):
    images = images.astype("float32")
    gray = (
        0.299 * images[..., 0]
        + 0.587 * images[..., 1]
        + 0.114 * images[..., 2]
    )
    return gray[..., np.newaxis]

(X_train_full_rgb, y_train_full), (X_test_rgb, y_test) = cifar10.load_data()

y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

all_indices = np.arange(len(X_train_full_rgb))

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full
)

X_train_rgb = X_train_full_rgb[train_indices]
X_val_rgb = X_train_full_rgb[val_indices]

y_train = y_train_full[train_indices]
y_val = y_train_full[val_indices]

# RGB -> Grayscale NTSC -> normalisasi.
X_train = rgb_to_ntsc_grayscale(
    X_train_rgb
) / 255.0

X_val = rgb_to_ntsc_grayscale(
    X_val_rgb
) / 255.0

X_test = rgb_to_ntsc_grayscale(
    X_test_rgb
) / 255.0

print("NTSC Training   :", X_train.shape, y_train.shape)
print("NTSC Validation :", X_val.shape, y_val.shape)
print("NTSC Test       :", X_test.shape, y_test.shape)

assert X_train.shape == (40000, 32, 32, 1)
assert X_val.shape == (10000, 32, 32, 1)
assert X_test.shape == (10000, 32, 32, 1)

NTSC Training   : (40000, 32, 32, 1) (40000,)
NTSC Validation : (10000, 32, 32, 1) (10000,)
NTSC Test       : (10000, 32, 32, 1) (10000,)


## 4. Load Trained Grayscale NTSC CNN

`compile=False` digunakan karena model hanya dipakai untuk inference / feature extraction.

In [4]:
tf.keras.backend.clear_session()

cnn_model = load_model(
    CNN_MODEL_PATH,
    compile=False
)

if FEATURE_LAYER not in [layer.name for layer in cnn_model.layers]:
    raise ValueError(
        f"Layer '{FEATURE_LAYER}' tidak ditemukan pada CNN."
    )

print("NTSC CNN berhasil dimuat.")
print("Input shape        :", cnn_model.input_shape)
print(
    "Feature layer shape:",
    cnn_model.get_layer(FEATURE_LAYER).output.shape
)

I0000 00:00:1791417947.290964    5182 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3672 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


NTSC CNN berhasil dimuat.
Input shape        : (None, 32, 32, 1)
Feature layer shape: (None, 512)


## 5. Buat NTSC CNN Feature Extractor

Output diambil dari layer `svm_features`.

```text
RGB image
    ↓
Grayscale NTSC
    ↓
NTSC CNN
    ↓
svm_features
    ↓
512-dimensional NTSC feature vector
```

In [5]:
feature_extractor = Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer(
        FEATURE_LAYER
    ).output,
    name="ntsc_cnn_feature_extractor"
)

feature_extractor.summary()

Model: "ntsc_cnn_feature_extractor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 32, 32, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 32, 32,    │        576 │ input_image[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 32, 32,    │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu           │ (None, 32, 32,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_conv1 │ (None, 32, 32,    │     36,864 │ stem_relu[0][0]   │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_bn1   │ (None, 32, 32,    │        256 │ stage1_block1_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_relu1 │ (None, 32, 32,    │          0 │ stage1_block1_bn… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_conv2 │ (None, 32, 32,    │     36,864 │ stage1_block1_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_bn2   │ (None, 32, 32,    │        256 │ stage1_block1_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_add   │ (None, 32, 32,    │          0 │ stage1_block1_bn… │
│ (Add)               │ 64)               │            │ stem_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_out   │ (None, 32, 32,    │          0 │ stage1_block1_ad… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_conv1 │ (None, 32, 32,    │     36,864 │ stage1_block1_ou… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_bn1   │ (None, 32, 32,    │        256 │ stage1_block2_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_relu1 │ (None, 32, 32,    │          0 │ stage1_block2_bn… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_conv2 │ (None, 32, 32,    │     36,864 │ stage1_block2_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_bn2   │ (None, 32, 32,    │        256 │ stage1_block2_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block2_add   │ (None, 32, 32,    │          0 │ stage1_block2_bn

 Total params: 11,441,472 (43.65 MB)

 Trainable params: 11,430,848 (43.61 MB)

 Non-trainable params: 10,624 (41.50 KB)

## 6. Extract NTSC Features

Feature extraction dilakukan untuk:
- training set,
- validation set,
- official test set.

Hasilnya menjadi matrix 512 feature per image.

> Jika VRAM penuh, shutdown kernel notebook training/testing lain sebelum menjalankan extraction.

In [6]:
BATCH_SIZE = 128

X_train_features = feature_extractor.predict(
    X_train,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_val_features = feature_extractor.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_test_features = feature_extractor.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=1
)

X_train_features = X_train_features.astype(np.float32)
X_val_features = X_val_features.astype(np.float32)
X_test_features = X_test_features.astype(np.float32)

print()
print("NTSC train features :", X_train_features.shape)
print("NTSC val features   :", X_val_features.shape)
print("NTSC test features  :", X_test_features.shape)

I0000 00:00:1791417949.173156    5312 service.cc:153] XLA service 0x70c4e0052ee0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791417949.173174    5312 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.26.0)
I0000 00:00:1791417949.199173    5312 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791417949.399263    5312 cuda_dnn.cc:461] Loaded cuDNN version 92600
W0000 00:00:1791417953.036088    5312 bfc_allocator.cc:383] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller b

  5/313 ━━━━━━━━━━━━━━━━━━━━ 11s 37ms/step

I0000 00:00:1791417955.272634    5312 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


313/313 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 5s 69ms/step
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step

NTSC train features : (40000, 512)
NTSC val features   : (10000, 512)
NTSC test features  : (10000, 512)


## 7. Validasi NTSC Feature Matrix

Layer `svm_features` menghasilkan 512-dimensional feature vector untuk setiap image.

In [7]:
FEATURE_DIM = int(
    cnn_model.get_layer(FEATURE_LAYER).output.shape[-1]
)

assert X_train_features.shape == (
    len(X_train),
    FEATURE_DIM
)

assert X_val_features.shape == (
    len(X_val),
    FEATURE_DIM
)

assert X_test_features.shape == (
    len(X_test),
    FEATURE_DIM
)

assert len(y_train) == len(X_train_features)
assert len(y_val) == len(X_val_features)
assert len(y_test) == len(X_test_features)

assert np.isfinite(X_train_features).all()
assert np.isfinite(X_val_features).all()
assert np.isfinite(X_test_features).all()

print("Feature dimension :", FEATURE_DIM)
print("Semua shape valid.")
print("Tidak ditemukan NaN/Inf pada extracted features.")

Feature dimension : 512
Semua shape valid.
Tidak ditemukan NaN/Inf pada extracted features.


## 8. Simpan ke `ntsc_features_v3.npz`

Archive menyimpan:
- `X_train_features`
- `X_val_features`
- `X_test_features`
- `y_train`
- `y_val`
- `y_test`
- `train_indices`
- `val_indices`
- metadata (`seed`, `feature_dim`, `feature_layer`, `representation`, `grayscale_formula`)

Indices dan labels memungkinkan kita memastikan RGB, AVG, dan NTSC benar-benar merepresentasikan sample yang sama sebelum feature concatenation.

In [8]:
np.savez_compressed(
    FEATURE_OUTPUT_PATH,
    X_train_features=X_train_features,
    X_val_features=X_val_features,
    X_test_features=X_test_features,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    train_indices=train_indices,
    val_indices=val_indices,
    seed=np.array(SEED, dtype=np.int64),
    feature_dim=np.array(FEATURE_DIM, dtype=np.int64),
    feature_layer=np.array(FEATURE_LAYER),
    representation=np.array("NTSC"),
    grayscale_formula=np.array("0.299R + 0.587G + 0.114B")
)

print("NTSC feature archive berhasil disimpan:")
print(FEATURE_OUTPUT_PATH)

NTSC feature archive berhasil disimpan:
features/ntsc_features_v3.npz


## 9. Verifikasi File Hasil

In [9]:
with np.load(FEATURE_OUTPUT_PATH) as avg:
    print("Keys:")
    for key in avg.files:
        print(
            f"- {key}:",
            avg[key].shape
            if avg[key].ndim > 0
            else avg[key].item()
        )

    assert np.array_equal(
        avg["train_indices"],
        train_indices
    )
    assert np.array_equal(
        avg["val_indices"],
        val_indices
    )
    assert np.array_equal(
        avg["y_train"],
        y_train
    )
    assert np.array_equal(
        avg["y_val"],
        y_val
    )
    assert np.array_equal(
        avg["y_test"],
        y_test
    )

print()
print("Verifikasi selesai: ntsc_features_v3.npz valid.")

Keys:
- X_train_features: (40000, 512)
- X_val_features: (10000, 512)
- X_test_features: (10000, 512)
- y_train: (40000,)
- y_val: (10000,)
- y_test: (10000,)
- train_indices: (40000,)
- val_indices: (10000,)
- seed: 30092026
- feature_dim: 512
- feature_layer: svm_features
- representation: NTSC
- grayscale_formula: 0.299R + 0.587G + 0.114B

Verifikasi selesai: ntsc_features_v3.npz valid.


# Selesai — NTSC Feature Extraction

Output:

```text
ntsc_features_v3.npz
```

Artifact Grayscale NTSC:

```text
cifar10_ntsc_cnn_v3.keras
cifar10_ntsc_svm_v3.joblib
ntsc_features_v3.npz
```

Pada tugas Feature Engineering nanti:

```text
rgb_features_v3.npz
        +
avg_features.npz
        +
ntsc_features_v3.npz
        ↓
RGB + AVG + NTSC feature fusion
```